# Advanced Problems: Cyclic Iterators

This notebook focuses on **cyclic iterators**: iterators that repeat a finite pattern indefinitely, or use cycling behavior to solve scheduling, pairing, windowing, and simulation problems.

The examples use best practices:

- prefer `itertools.cycle` when it already solves the problem;
- use `itertools.islice` to inspect only a finite part of an infinite iterator;
- avoid materializing infinite iterators;
- test edge cases such as empty inputs and one-shot iterables;
- separate problem statements from solutions.


## Setup


In [1]:
from __future__ import annotations

from collections import deque
from itertools import cycle, islice, count
from typing import Any, Generic, Iterable, Iterator, Sequence, TypeVar

T = TypeVar("T")
U = TypeVar("U")


## Problem 1 — Build a production-quality cyclic iterator

Implement a class `CycleCursor` that cycles over a finite iterable.

Requirements:

1. Accept any finite iterable, not only lists or strings.
2. Raise `ValueError` when initialized with an empty iterable.
3. Implement the iterator protocol.
4. Track how many values have been produced using a `.steps` property.
5. Support `.reset()` to move back to the original start position.
6. Support an optional `start` index.
7. Provide a helpful `repr`.

Example:

```python
c = CycleCursor("NSWE", start=2)
[next(c) for _ in range(6)]
# ['W', 'E', 'N', 'S', 'W', 'E']
```


In [2]:
class CycleCursor(Generic[T]):
    """A resettable cyclic iterator over a finite snapshot of values."""

    def __init__(self, iterable: Iterable[T], start: int = 0):
        self._items = tuple(iterable)
        if not self._items:
            raise ValueError("CycleCursor requires at least one item")

        self._start = start % len(self._items)
        self._index = self._start
        self._steps = 0

    def __iter__(self) -> "CycleCursor[T]":
        return self

    def __next__(self) -> T:
        value = self._items[self._index]
        self._index = (self._index + 1) % len(self._items)
        self._steps += 1
        return value

    @property
    def steps(self) -> int:
        return self._steps

    def reset(self) -> None:
        self._index = self._start
        self._steps = 0

    def __repr__(self) -> str:
        return (
            f"{type(self).__name__}"
            f"(items={self._items!r}, start={self._start}, "
            f"index={self._index}, steps={self._steps})"
        )


directions = CycleCursor("NSWE", start=2)
list(islice(directions, 6))


['W', 'E', 'N', 'S', 'W', 'E']

In [3]:
# Tests
c = CycleCursor("ABC")
assert [next(c) for _ in range(8)] == list("ABCABCAB")
assert c.steps == 8

c.reset()
assert c.steps == 0
assert [next(c) for _ in range(4)] == list("ABCA")

c = CycleCursor(["red", "green", "blue"], start=4)
assert [next(c) for _ in range(5)] == ["green", "blue", "red", "green", "blue"]

try:
    CycleCursor([])
except ValueError as ex:
    assert "at least one item" in str(ex)
else:
    raise AssertionError("Expected ValueError for empty iterable")

print("Problem 1 tests passed.")


Problem 1 tests passed.


## Problem 2 — Pair a finite stream with an infinite cycle

Write a function `label_items(items, labels)` that returns strings combining each item with a cycling label.

Requirements:

1. Use `itertools.cycle`.
2. Return a list.
3. Validate that `labels` is not empty.
4. Do not repeat labels manually using multiplication.
5. Work for any finite iterable of `items`.

Example:

```python
label_items(range(1, 11), "NSWE")
# ['1N', '2S', '3W', '4E', '5N', '6S', '7W', '8E', '9N', '10S']
```


In [4]:
def label_items(items: Iterable[Any], labels: Iterable[Any]) -> list[str]:
    label_snapshot = tuple(labels)
    if not label_snapshot:
        raise ValueError("labels must not be empty")

    return [f"{item}{label}" for item, label in zip(items, cycle(label_snapshot))]


label_items(range(1, 11), "NSWE")


['1N', '2S', '3W', '4E', '5N', '6S', '7W', '8E', '9N', '10S']

In [5]:
# More examples
assert label_items(range(1, 11), "NSWE") == [
    "1N", "2S", "3W", "4E", "5N", "6S", "7W", "8E", "9N", "10S"
]

assert label_items(["task-1", "task-2", "task-3", "task-4"], ["low", "high"]) == [
    "task-1low", "task-2high", "task-3low", "task-4high"
]

assert label_items([], "ABC") == []

try:
    label_items([1, 2, 3], [])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for empty labels")

print("Problem 2 tests passed.")


Problem 2 tests passed.


## Problem 3 — Generate a lazy cyclic code stream

Create a generator function `cyclic_codes(prefixes, numbers)` that lazily combines a finite sequence of prefixes with a sequence of numbers.

Requirements:

1. `prefixes` cycles forever.
2. `numbers` may be finite or infinite.
3. Stop when `numbers` is exhausted.
4. Validate that `prefixes` is not empty.
5. Return an iterator, not a list.

Example:

```python
codes = cyclic_codes("ABC", range(1, 8))
list(codes)
# ['A-1', 'B-2', 'C-3', 'A-4', 'B-5', 'C-6', 'A-7']
```


In [6]:
def cyclic_codes(prefixes: Iterable[str], numbers: Iterable[int]) -> Iterator[str]:
    prefix_snapshot = tuple(prefixes)
    if not prefix_snapshot:
        raise ValueError("prefixes must not be empty")

    for prefix, number in zip(cycle(prefix_snapshot), numbers):
        yield f"{prefix}-{number}"


list(cyclic_codes("ABC", range(1, 8)))


['A-1', 'B-2', 'C-3', 'A-4', 'B-5', 'C-6', 'A-7']

In [7]:
# Demonstrate laziness with an infinite number stream.
first_12 = list(islice(cyclic_codes("XY", count(100)), 12))
assert first_12 == [
    "X-100", "Y-101", "X-102", "Y-103", "X-104", "Y-105",
    "X-106", "Y-107", "X-108", "Y-109", "X-110", "Y-111"
]

# Finite number stream ends normally.
assert list(cyclic_codes(["Q"], range(3))) == ["Q-0", "Q-1", "Q-2"]

try:
    next(cyclic_codes([], range(3)))
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for empty prefixes")

print("Problem 3 tests passed.")


Problem 3 tests passed.


## Problem 4 — Round-robin scheduler with budgets

You have workers and tasks. Each task has a cost. Each worker has a budget. Assign tasks in round-robin order, but skip workers who cannot afford the current task.

Implement `assign_tasks_round_robin(workers, budgets, tasks)`.

Inputs:

- `workers`: sequence of worker names
- `budgets`: dictionary mapping worker name to remaining budget
- `tasks`: sequence of `(task_name, cost)` pairs

Rules:

1. Workers are considered cyclically.
2. A task goes to the first worker, in cyclic order, who has enough remaining budget.
3. After assignment, subtract cost from that worker's budget.
4. If no worker can afford a task after one full rotation, mark it as unassigned.
5. Return `(assignments, remaining_budgets)`.
6. Do not mutate the original `budgets` dictionary.

This is a realistic use case for cyclic iterators because the cycle is not blindly consumed; it must cooperate with business rules.


In [8]:
def assign_tasks_round_robin(
    workers: Sequence[str],
    budgets: dict[str, int],
    tasks: Sequence[tuple[str, int]],
) -> tuple[list[tuple[str, str | None, int]], dict[str, int]]:
    if not workers:
        raise ValueError("workers must not be empty")

    remaining = dict(budgets)
    missing = [worker for worker in workers if worker not in remaining]
    if missing:
        raise KeyError(f"Missing budgets for workers: {missing}")

    worker_cycle = cycle(workers)
    assignments: list[tuple[str, str | None, int]] = []

    for task_name, cost in tasks:
        assigned_worker: str | None = None

        # At most one full rotation. This prevents an infinite loop.
        for _ in range(len(workers)):
            worker = next(worker_cycle)
            if remaining[worker] >= cost:
                remaining[worker] -= cost
                assigned_worker = worker
                break

        assignments.append((task_name, assigned_worker, cost))

    return assignments, remaining


workers = ["Ada", "Linus", "Grace"]
budgets = {"Ada": 8, "Linus": 5, "Grace": 10}
tasks = [("A", 3), ("B", 4), ("C", 7), ("D", 6), ("E", 2), ("F", 8)]

assign_tasks_round_robin(workers, budgets, tasks)


([('A', 'Ada', 3),
  ('B', 'Linus', 4),
  ('C', 'Grace', 7),
  ('D', None, 6),
  ('E', 'Ada', 2),
  ('F', None, 8)],
 {'Ada': 3, 'Linus': 1, 'Grace': 3})

In [9]:
# Tests
workers = ["Ada", "Linus", "Grace"]
budgets = {"Ada": 8, "Linus": 5, "Grace": 10}
tasks = [("A", 3), ("B", 4), ("C", 7), ("D", 6), ("E", 2), ("F", 8)]

assignments, remaining = assign_tasks_round_robin(workers, budgets, tasks)

assert assignments == [
    ("A", "Ada", 3),
    ("B", "Linus", 4),
    ("C", "Grace", 7),
    ("D", None, 6),
    ("E", "Ada", 2),
    ("F", None, 8),
]

assert remaining == {"Ada": 3, "Linus": 1, "Grace": 3}
assert budgets == {"Ada": 8, "Linus": 5, "Grace": 10}  # Original was not mutated.

try:
    assign_tasks_round_robin([], {}, tasks)
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for empty workers")

print("Problem 4 tests passed.")


Problem 4 tests passed.


## Problem 5 — Cyclic windows over a circular sequence

Implement `cyclic_windows(values, size, limit=None)`.

The function returns windows of length `size`, wrapping around the sequence cyclically.

Rules:

1. `values` must be non-empty.
2. `size` must be positive.
3. If `limit is None`, produce exactly `len(values)` windows.
4. If `limit` is provided, produce exactly `limit` windows.
5. Return a list of tuples.

Example:

```python
cyclic_windows([1, 2, 3, 4], size=3)
# [(1, 2, 3), (2, 3, 4), (3, 4, 1), (4, 1, 2)]
```


In [10]:
def cyclic_windows(values: Sequence[T], size: int, limit: int | None = None) -> list[tuple[T, ...]]:
    if not values:
        raise ValueError("values must not be empty")
    if size <= 0:
        raise ValueError("size must be positive")

    window_count = len(values) if limit is None else limit
    if window_count < 0:
        raise ValueError("limit must not be negative")

    result: list[tuple[T, ...]] = []
    n = len(values)

    for start in range(window_count):
        window = tuple(values[(start + offset) % n] for offset in range(size))
        result.append(window)

    return result


cyclic_windows([1, 2, 3, 4], size=3)


[(1, 2, 3), (2, 3, 4), (3, 4, 1), (4, 1, 2)]

In [11]:
# More examples and tests
assert cyclic_windows([1, 2, 3, 4], size=3) == [
    (1, 2, 3),
    (2, 3, 4),
    (3, 4, 1),
    (4, 1, 2),
]

assert cyclic_windows("AB", size=5, limit=4) == [
    ("A", "B", "A", "B", "A"),
    ("B", "A", "B", "A", "B"),
    ("A", "B", "A", "B", "A"),
    ("B", "A", "B", "A", "B"),
]

assert cyclic_windows([42], size=4) == [(42, 42, 42, 42)]

try:
    cyclic_windows([], size=2)
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for empty values")

print("Problem 5 tests passed.")


Problem 5 tests passed.


## Problem 6 — Weighted cyclic scheduler

Implement `weighted_cycle(items, weights)`.

It should cycle through `items` forever, but each item appears according to its integer weight.

Example:

```python
stream = weighted_cycle(["A", "B", "C"], [1, 3, 2])
list(islice(stream, 12))
# ['A', 'B', 'B', 'B', 'C', 'C', 'A', 'B', 'B', 'B', 'C', 'C']
```

Requirements:

1. `items` and `weights` must have the same length.
2. Negative weights are invalid.
3. Zero-weight items should be skipped.
4. At least one weight must be positive.
5. The function must be lazy and infinite.

Best-practice note: avoid validating length with only `zip(items, weights)`, because `zip` silently truncates to the shorter iterable.


In [12]:
def weighted_cycle(items: Iterable[T], weights: Iterable[int]) -> Iterator[T]:
    item_snapshot = tuple(items)
    weight_snapshot = tuple(weights)

    if len(item_snapshot) != len(weight_snapshot):
        raise ValueError("items and weights must have the same length")
    if any(weight < 0 for weight in weight_snapshot):
        raise ValueError("weights must not be negative")

    positive_pairs = [
        (item, weight)
        for item, weight in zip(item_snapshot, weight_snapshot)
        if weight > 0
    ]

    if not positive_pairs:
        raise ValueError("at least one weight must be positive")

    while True:
        for item, weight in positive_pairs:
            for _ in range(weight):
                yield item


stream = weighted_cycle(["A", "B", "C"], [1, 3, 2])
list(islice(stream, 12))


['A', 'B', 'B', 'B', 'C', 'C', 'A', 'B', 'B', 'B', 'C', 'C']

In [13]:
# Tests
assert list(islice(weighted_cycle(["A", "B", "C"], [1, 3, 2]), 12)) == [
    "A", "B", "B", "B", "C", "C",
    "A", "B", "B", "B", "C", "C",
]

assert list(islice(weighted_cycle(["silent", "active"], [0, 2]), 5)) == [
    "active", "active", "active", "active", "active"
]

for bad_items, bad_weights in [
    (["A", "B"], [1]),
    (["A"], [-1]),
    (["A", "B"], [0, 0]),
]:
    try:
        list(islice(weighted_cycle(bad_items, bad_weights), 3))
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError for invalid weighted cycle input")

print("Problem 6 tests passed.")


Problem 6 tests passed.


## Problem 7 — Recreate `itertools.cycle` lazily for one-shot iterables

The standard library already has `itertools.cycle`, but implementing a simplified version is an excellent exercise.

Implement `lazy_cycle(iterable)`.

Requirements:

1. It should work with one-shot iterables such as generators.
2. During the first pass, yield values as they arrive and cache them.
3. After the source iterable is exhausted, cycle over the cache forever.
4. If the source iterable is empty, the generator should simply stop.
5. Use `islice` in tests to avoid infinite loops.

This mirrors the key idea behind cycling a generator: the values must be remembered after they have been seen once.


In [14]:
def lazy_cycle(iterable: Iterable[T]) -> Iterator[T]:
    saved: list[T] = []

    for item in iterable:
        yield item
        saved.append(item)

    if not saved:
        return

    while True:
        for item in saved:
            yield item


def one_shot_numbers() -> Iterator[int]:
    print("source started")
    for number in [10, 20, 30]:
        print(f"source yielded {number}")
        yield number


stream = lazy_cycle(one_shot_numbers())
list(islice(stream, 8))


source started
source yielded 10
source yielded 20
source yielded 30


[10, 20, 30, 10, 20, 30, 10, 20]

In [15]:
# Tests
assert list(islice(lazy_cycle([1, 2, 3]), 10)) == [1, 2, 3, 1, 2, 3, 1, 2, 3, 1]

source = (x * 10 for x in range(1, 4))
assert list(islice(lazy_cycle(source), 8)) == [10, 20, 30, 10, 20, 30, 10, 20]

assert list(lazy_cycle([])) == []

print("Problem 7 tests passed.")


Problem 7 tests passed.


## Problem 8 — Round-robin merge of uneven finite iterables

Implement `round_robin_merge(*iterables)`.

It should yield one item from each iterable in cyclic order. When an iterable is exhausted, remove it from the rotation. Continue until all iterables are exhausted.

Example:

```python
list(round_robin_merge("ABC", [1, 2], ["x", "y", "z", "w"]))
# ['A', 1, 'x', 'B', 2, 'y', 'C', 'z', 'w']
```

Requirements:

1. Return an iterator.
2. Do not convert the full inputs to lists.
3. Remove exhausted iterators cleanly.
4. Preserve the order within each input.
5. Handle zero inputs.


In [16]:
def round_robin_merge(*iterables: Iterable[T]) -> Iterator[T]:
    active = deque(iter(iterable) for iterable in iterables)

    while active:
        current = active.popleft()

        try:
            value = next(current)
        except StopIteration:
            continue

        yield value
        active.append(current)


list(round_robin_merge("ABC", [1, 2], ["x", "y", "z", "w"]))


['A', 1, 'x', 'B', 2, 'y', 'C', 'z', 'w']

In [17]:
# Tests
assert list(round_robin_merge("ABC", [1, 2], ["x", "y", "z", "w"])) == [
    "A", 1, "x",
    "B", 2, "y",
    "C", "z", "w",
]

assert list(round_robin_merge()) == []
assert list(round_robin_merge([], [1, 2], [])) == [1, 2]
assert list(round_robin_merge((x for x in [10, 20]), "ABCD")) == [10, "A", 20, "B", "C", "D"]

print("Problem 8 tests passed.")


Problem 8 tests passed.


## Problem 9 — Cycle-aware simulation: rotating compass robot

A robot starts at a coordinate and a compass direction. It receives commands:

- `"F"`: move one step forward in the current direction
- `"L"`: turn left
- `"R"`: turn right

Implement `simulate_robot(commands, start=(0, 0), direction="N")`.

Requirements:

1. Use cyclic indexing over compass directions.
2. Validate the starting direction.
3. Return a list of positions after each command, including turns.
4. Turns do not change the position.
5. Use clear constants for direction order and movement vectors.

Example:

```python
simulate_robot("FFRFFLFF")
# [(0, 1), (0, 2), (0, 2), (1, 2), (2, 2), (2, 2), (2, 3), (2, 4)]
```


In [18]:
DIRECTIONS = ("N", "E", "S", "W")
MOVE = {
    "N": (0, 1),
    "E": (1, 0),
    "S": (0, -1),
    "W": (-1, 0),
}


def simulate_robot(
    commands: Iterable[str],
    start: tuple[int, int] = (0, 0),
    direction: str = "N",
) -> list[tuple[int, int]]:
    if direction not in DIRECTIONS:
        raise ValueError(f"Unknown direction: {direction!r}")

    x, y = start
    direction_index = DIRECTIONS.index(direction)
    positions: list[tuple[int, int]] = []

    for command in commands:
        if command == "L":
            direction_index = (direction_index - 1) % len(DIRECTIONS)
        elif command == "R":
            direction_index = (direction_index + 1) % len(DIRECTIONS)
        elif command == "F":
            dx, dy = MOVE[DIRECTIONS[direction_index]]
            x += dx
            y += dy
        else:
            raise ValueError(f"Unknown command: {command!r}")

        positions.append((x, y))

    return positions


simulate_robot("FFRFFLFF")


[(0, 1), (0, 2), (0, 2), (1, 2), (2, 2), (2, 2), (2, 3), (2, 4)]

In [19]:
# Tests
assert simulate_robot("FFRFFLFF") == [
    (0, 1), (0, 2), (0, 2), (1, 2),
    (2, 2), (2, 2), (2, 3), (2, 4)
]

assert simulate_robot("RFRFRFRF") == [
    (0, 0), (1, 0), (1, 0), (1, -1),
    (1, -1), (0, -1), (0, -1), (0, 0)
]

try:
    simulate_robot("X")
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for unknown command")

print("Problem 9 tests passed.")


Problem 9 tests passed.


## Problem 10 — Mini-project: cyclic batch assignment with audit rows

Build `assign_batches(records, reviewers, batch_size)`.

A record is any object. Reviewers cycle round-robin. Records are grouped into batches of size `batch_size`. Each batch is assigned to the next reviewer.

Return a list of dictionaries with these fields:

- `"batch_id"`: starting at 1
- `"reviewer"`
- `"records"`
- `"record_count"`

Requirements:

1. Validate that `reviewers` is non-empty.
2. Validate that `batch_size` is positive.
3. Do not mutate the input records.
4. The final batch may be smaller than `batch_size`.
5. Use `cycle` for reviewer rotation.
6. Use a helper function for chunking.


In [20]:
def chunks(values: Sequence[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    for start in range(0, len(values), size):
        yield tuple(values[start:start + size])


def assign_batches(
    records: Sequence[T],
    reviewers: Iterable[str],
    batch_size: int,
) -> list[dict[str, Any]]:
    reviewer_snapshot = tuple(reviewers)
    if not reviewer_snapshot:
        raise ValueError("reviewers must not be empty")
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    reviewer_cycle = cycle(reviewer_snapshot)
    audit_rows: list[dict[str, Any]] = []

    for batch_id, batch in enumerate(chunks(records, batch_size), start=1):
        audit_rows.append(
            {
                "batch_id": batch_id,
                "reviewer": next(reviewer_cycle),
                "records": batch,
                "record_count": len(batch),
            }
        )

    return audit_rows


assign_batches(list(range(1, 11)), reviewers=["Ada", "Linus", "Grace"], batch_size=3)


[{'batch_id': 1, 'reviewer': 'Ada', 'records': (1, 2, 3), 'record_count': 3},
 {'batch_id': 2, 'reviewer': 'Linus', 'records': (4, 5, 6), 'record_count': 3},
 {'batch_id': 3, 'reviewer': 'Grace', 'records': (7, 8, 9), 'record_count': 3},
 {'batch_id': 4, 'reviewer': 'Ada', 'records': (10,), 'record_count': 1}]

In [21]:
# Tests
rows = assign_batches(list(range(1, 11)), reviewers=["Ada", "Linus", "Grace"], batch_size=3)

assert rows == [
    {"batch_id": 1, "reviewer": "Ada", "records": (1, 2, 3), "record_count": 3},
    {"batch_id": 2, "reviewer": "Linus", "records": (4, 5, 6), "record_count": 3},
    {"batch_id": 3, "reviewer": "Grace", "records": (7, 8, 9), "record_count": 3},
    {"batch_id": 4, "reviewer": "Ada", "records": (10,), "record_count": 1},
]

assert assign_batches([], reviewers=["Ada"], batch_size=3) == []

for bad_reviewers, bad_batch_size in [
    ([], 3),
    (["Ada"], 0),
]:
    try:
        assign_batches([1, 2, 3], reviewers=bad_reviewers, batch_size=bad_batch_size)
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError for invalid batch assignment input")

print("Problem 10 tests passed.")


Problem 10 tests passed.


## Summary checklist

When working with cyclic iterators:

- Use `cycle(pattern)` for infinite repetition of a known finite pattern.
- Use `islice(infinite_iterator, n)` to safely inspect a finite prefix.
- Validate empty patterns before cycling when your logic requires at least one value.
- Avoid `list(infinite_iterator)`, `sum(infinite_iterator)`, or unbounded loops.
- Snapshot inputs when you need stable behavior.
- Be careful with `zip`: it stops at the shortest iterable.
- In scheduling problems, limit each search to one full rotation when "nobody can accept this item" is possible.
